In [17]:
# !pip install torch scikit-learn joblib fastapi uvicorn nest-asyncio pyngrok pandas numpy xgboost lightgbm --quiet

In [18]:
# from google.colab import files
# import pandas as pd
# import io

# uploaded = files.upload()
# filename = list(uploaded.keys())[0]
# df_uni = pd.read_csv(io.BytesIO(uploaded[filename]))

# df_uni = df_uni.rename(columns=lambda x: x.strip())
# print("Shape:", df_uni.shape)
# print("Columns:", df_uni.columns.tolist())
# df_uni.head()

In [19]:
import numpy as np

df_uni = df_uni.rename(columns=lambda x: x.strip())

# Drop garbage rows if any
df_uni = df_uni[~df_uni["university_name"].str.contains(
    "Category|portal|states", case=False, na=False
)].copy()

# Type coercion
num_cols = ["min_ielts", "ielts_band", "toefl", "duolingo", "gre_score",
            "work_exp", "qs_rank", "min_cgpa", "tuition_usd"]
for col in num_cols:
    df_uni[col] = pd.to_numeric(df_uni[col], errors="coerce")

df_uni["gre_required"] = df_uni["gre"].astype(int)  # binary flag from dataset

# Fill nulls
df_uni = df_uni.fillna({
    "min_ielts": 6.5, "ielts_band": 6.5, "toefl": 85,
    "duolingo": 110, "gre_score": 0, "work_exp": 0,
    "min_cgpa": 3.0, "tuition_usd": 20000
})

# ── Engineered university features ──────────────────────────────────────────
rank_max = df_uni["qs_rank"].max()
df_uni["rank_score"]        = 1 - (df_uni["qs_rank"] / rank_max)           # 0–1, higher = better ranked
df_uni["rank_tier"]         = pd.qcut(df_uni["qs_rank"], q=5,               # 1=elite … 5=accessible
                                       labels=[1,2,3,4,5]).astype(int)
df_uni["strictness_score"]  = (                                              # how hard to get in
    df_uni["min_cgpa"] / 4.0 * 0.4 +
    df_uni["min_ielts"] / 9.0 * 0.3 +
    df_uni["gre_required"]    * 0.3
)
df_uni["affordability"]     = 1 - (df_uni["tuition_usd"] /
                                    df_uni["tuition_usd"].max())

print("Cleaned shape:", df_uni.shape)
print("Countries:", df_uni["country"].value_counts().to_dict())
df_uni.head(3)

Cleaned shape: (216, 18)
Countries: {'UK': 89, 'Germany': 49, 'Australia': 36, 'Canada': 29, 'Qatar': 13}


,university_name,country,qs_rank,min_ielts,ielts_band,toefl,duolingo,gre,work_exp,source,min_cgpa,gre_score,tuition_usd,gre_required,rank_score,rank_tier,strictness_score,affordability
0,Imperial College London,UK,2,6.5,7.0,100,120,1,0,groq,3.7,320,42000,1,0.998335,1,0.886667,0.275862
1,University of Oxford,UK,4,7.0,7.0,110,125,1,0,groq,3.7,325,40000,1,0.996669,1,0.903333,0.310345
2,University of Cambridge,UK,6,7.0,7.0,110,125,1,1,groq,3.7,325,40000,1,0.995004,1,0.903333,0.310345


In [20]:
import numpy as np
import pandas as pd

np.random.seed(42)
N_STUDENTS = 20000

students = pd.DataFrame({
    "IELTS":     np.round(np.random.uniform(4.5, 9.0, N_STUDENTS), 1),
    "IELTS_BAND":np.round(np.random.uniform(4.5, 9.0, N_STUDENTS), 1),
    "TOEFL":     np.random.randint(45, 120, N_STUDENTS),
    "DUOLINGO":  np.random.randint(60, 160, N_STUDENTS),
    "GRE_SCORE": np.where(np.random.rand(N_STUDENTS) < 0.4,
                          np.random.randint(280, 340, N_STUDENTS), 0),
    "WORK_EXP":  np.random.randint(0, 8, N_STUDENTS),
    "CGPA":      np.round(np.random.uniform(1.8, 4.0, N_STUDENTS), 2),
})


def compute_label(stu, uni):
    """
    Probabilistic label — returns probability of admission.
    Uses continuous gap scores so the NN learns smooth decision surfaces.
    """
    # ── Hard disqualifiers ─────────────────────────────────────────────
    cgpa_gap  = stu["CGPA"]   - uni["min_cgpa"]
    ielts_gap = stu["IELTS"]  - uni["min_ielts"]

    # Soft rejection zone: very far below → almost certainly 0
    if cgpa_gap < -0.8 or ielts_gap < -1.5:
        return 0

    # ── Component scores (each –1 → +1) ────────────────────────────────
    def sigmoid_gap(gap, scale=1.0):
        return 2 / (1 + np.exp(-gap / scale)) - 1   # centred sigmoid

    s_cgpa  = sigmoid_gap(cgpa_gap,  scale=0.3) * 0.30
    s_ielts = sigmoid_gap(ielts_gap, scale=0.5) * 0.25

    # TOEFL
    toefl_gap = stu["TOEFL"] - uni["toefl"]
    s_toefl = sigmoid_gap(toefl_gap, scale=5) * 0.15

    # Duolingo
    duo_gap = stu["DUOLINGO"] - uni["duolingo"]
    s_duo   = sigmoid_gap(duo_gap, scale=5) * 0.10

    # GRE
    if uni["gre_required"] == 1 and uni["gre_score"] > 0:
        gre_gap = stu["GRE_SCORE"] - uni["gre_score"]
        s_gre   = sigmoid_gap(gre_gap, scale=10) * 0.12
        if stu["GRE_SCORE"] == 0:
            s_gre = -0.20   # hard penalty for missing required GRE
    else:
        # GRE not required but having one is a bonus
        s_gre = 0.03 if stu["GRE_SCORE"] > 300 else 0.0

    # Work exp
    if uni["work_exp"] > 0:
        we_gap = stu["WORK_EXP"] - uni["work_exp"]
        s_we   = sigmoid_gap(we_gap, scale=1) * 0.08
    else:
        s_we   = 0.03 if stu["WORK_EXP"] >= 1 else 0.0

    total = s_cgpa + s_ielts + s_toefl + s_duo + s_gre + s_we

    # rank difficulty penalty (top-10 unis are much harder)
    rank_penalty = 0.15 * (1 - uni["rank_score"])
    total -= rank_penalty

    # convert to probability with noise
    prob = 1 / (1 + np.exp(-total * 3))
    noise = np.random.normal(0, 0.04)
    prob = np.clip(prob + noise, 0, 1)

    return int(prob >= 0.5)


# ── Build pairs ──────────────────────────────────────────────────────────────
print("Building training pairs… (this takes ~30 sec)")
pairs = []
for _, uni in df_uni.iterrows():
    sample = students.sample(180, replace=True)
    for _, stu in sample.iterrows():
        label = compute_label(stu, uni)
        pairs.append({
            # student features
            "IELTS":          stu["IELTS"],
            "IELTS_BAND":     stu["IELTS_BAND"],
            "TOEFL":          stu["TOEFL"],
            "DUOLINGO":       stu["DUOLINGO"],
            "GRE_SCORE":      stu["GRE_SCORE"],
            "HAS_GRE":        int(stu["GRE_SCORE"] > 0),
            "WORK_EXP":       stu["WORK_EXP"],
            "CGPA":           stu["CGPA"],
            # gap features — key for the model!
            "CGPA_GAP":       stu["CGPA"]      - uni["min_cgpa"],
            "IELTS_GAP":      stu["IELTS"]     - uni["min_ielts"],
            "TOEFL_GAP":      stu["TOEFL"]     - uni["toefl"],
            "DUOLINGO_GAP":   stu["DUOLINGO"]  - uni["duolingo"],
            "GRE_GAP":        stu["GRE_SCORE"] - uni["gre_score"] if uni["gre_required"] else 0,
            "WORK_EXP_GAP":   stu["WORK_EXP"]  - uni["work_exp"],
            # university features
            "UNI_IELTS":      uni["min_ielts"],
            "UNI_IELTS_BAND": uni["ielts_band"],
            "UNI_TOEFL":      uni["toefl"],
            "UNI_DUOLINGO":   uni["duolingo"],
            "UNI_GRE":        uni["gre_score"],
            "UNI_GRE_REQ":    uni["gre_required"],
            "UNI_WORKEXP":    uni["work_exp"],
            "UNI_CGPA":       uni["min_cgpa"],
            "UNI_RANK":       uni["qs_rank"],
            "UNI_RANK_SCORE": uni["rank_score"],
            "UNI_RANK_TIER":  uni["rank_tier"],
            "UNI_STRICTNESS": uni["strictness_score"],
            "UNI_TUITION":    uni["tuition_usd"],
            "LABEL":          label,
        })

train_df = pd.DataFrame(pairs)
print(f"Training pairs: {train_df.shape}")
print(f"Label balance: {train_df['LABEL'].value_counts().to_dict()}")

Building training pairs… (this takes ~30 sec)
Training pairs: (38880, 28)
Label balance: {0: 22092, 1: 16788}


In [21]:
import torch
import torch.nn as nn
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

FEATURE_COLS = [c for c in train_df.columns if c != "LABEL"]

X = train_df[FEATURE_COLS].values
y = train_df["LABEL"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

joblib.dump(scaler, "scaler.pkl")
joblib.dump(FEATURE_COLS, "feature_cols.pkl")

# ── 1. XGBoost ───────────────────────────────────────────────────────────────
print("Training XGBoost…")
xgb = XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    use_label_encoder=False, eval_metric="logloss",
    random_state=42, n_jobs=-1
)
xgb.fit(X_train_sc, y_train,
        eval_set=[(X_test_sc, y_test)], verbose=False)
joblib.dump(xgb, "xgb_model.pkl")

xgb_prob_test = xgb.predict_proba(X_test_sc)[:, 1]
print(f"XGBoost AUC: {roc_auc_score(y_test, xgb_prob_test):.4f}")

# ── 2. Neural Network ────────────────────────────────────────────────────────
class EligibilityNet(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, 128),        nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, 64),         nn.BatchNorm1d(64),  nn.GELU(), nn.Dropout(0.1),
            nn.Linear(64, 32),          nn.GELU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nTraining Neural Network on {device}…")

X_tr_t = torch.tensor(X_train_sc, dtype=torch.float32).to(device)
y_tr_t = torch.tensor(y_train,    dtype=torch.float32).to(device)
X_te_t = torch.tensor(X_test_sc,  dtype=torch.float32).to(device)

net = EligibilityNet(X_train_sc.shape[1]).to(device)
optimizer = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=80)

pos_weight = torch.tensor([(y_train == 0).sum() / max((y_train == 1).sum(), 1)],
                           dtype=torch.float32).to(device)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

dataset   = torch.utils.data.TensorDataset(X_tr_t, y_tr_t.unsqueeze(1))
loader    = torch.utils.data.DataLoader(dataset, batch_size=1024, shuffle=True)

best_auc, patience_ctr = 0, 0
for epoch in range(1, 121):
    net.train()
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(net(xb), yb)
        loss.backward()
        optimizer.step()
    scheduler.step()

    if epoch % 10 == 0:
        net.eval()
        with torch.no_grad():
            preds = torch.sigmoid(net(X_te_t)).cpu().numpy().ravel()
        auc = roc_auc_score(y_test, preds)
        print(f"  Epoch {epoch:3d} | AUC: {auc:.4f}")
        if auc > best_auc:
            best_auc = auc
            torch.save(net.state_dict(), "best_net.pt")
            patience_ctr = 0
        else:
            patience_ctr += 1
        if patience_ctr >= 4:
            print("  Early stopping.")
            break

net.load_state_dict(torch.load("best_net.pt"))
net.eval()

# ── Final Evaluation ─────────────────────────────────────────────────────────
with torch.no_grad():
    nn_probs = torch.sigmoid(net(X_te_t)).cpu().numpy().ravel()

xgb_probs = xgb.predict_proba(X_test_sc)[:, 1]
ensemble_probs = 0.45 * nn_probs + 0.55 * xgb_probs
ensemble_preds = (ensemble_probs >= 0.5).astype(int)

print(f"\n{'='*50}")
print(f"Neural Net  AUC : {roc_auc_score(y_test, nn_probs):.4f}")
print(f"XGBoost     AUC : {roc_auc_score(y_test, xgb_probs):.4f}")
print(f"ENSEMBLE    AUC : {roc_auc_score(y_test, ensemble_probs):.4f}")
print(f"\nEnsemble Classification Report:")
print(classification_report(y_test, ensemble_preds, target_names=["Not Eligible","Eligible"]))

Training XGBoost…


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [09:52:39] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost AUC: 0.9936

Training Neural Network on cuda…
  Epoch  10 | AUC: 0.9884
  Epoch  20 | AUC: 0.9912
  Epoch  30 | AUC: 0.9922
  Epoch  40 | AUC: 0.9926
  Epoch  50 | AUC: 0.9930
  Epoch  60 | AUC: 0.9931
  Epoch  70 | AUC: 0.9932
  Epoch  80 | AUC: 0.9932
  Epoch  90 | AUC: 0.9932
  Epoch 100 | AUC: 0.9932
  Epoch 110 | AUC: 0.9932
  Epoch 120 | AUC: 0.9933

Neural Net  AUC : 0.9933
XGBoost     AUC : 0.9936
ENSEMBLE    AUC : 0.9940

Ensemble Classification Report:
              precision    recall  f1-score   support

Not Eligible       0.97      0.95      0.96      3314
    Eligible       0.94      0.96      0.95      2518

    accuracy                           0.95      5832
   macro avg       0.95      0.95      0.95      5832
weighted avg       0.95      0.95      0.95      5832



In [22]:
import pandas as pd
import numpy as np
import torch, joblib

scaler       = joblib.load("scaler.pkl")
FEATURE_COLS = joblib.load("feature_cols.pkl")
xgb          = joblib.load("xgb_model.pkl")
net.load_state_dict(torch.load("best_net.pt"))
net.eval()

COUNTRY_EMOJI = {"UK":"🇬🇧","Germany":"🇩🇪","Australia":"🇦🇺","Canada":"🇨🇦","Qatar":"🇶🇦"}

SCORE_BOUNDS = {
    "IELTS":(0.0,9.0), "TOEFL":(0,120), "DUOLINGO":(10,160),
    "GRE_SCORE":(0,340), "WORK_EXP":(0,40), "CGPA":(0.0,4.0),
}

TIER_ORDER = {"✅ Safe": 0, "🎯 Match": 1, "🌟 Dream": 2, "❌ Unlikely": 3}

def validate_and_clamp(user):
    cleaned, errors = dict(user), []
    for field, (lo, hi) in SCORE_BOUNDS.items():
        if field not in cleaned: continue
        val = cleaned[field]
        if val < lo or val > hi:
            errors.append(f"{field} must be between {lo} and {hi} (got {val})")
        cleaned[field] = max(lo, min(hi, val))
    if errors: raise ValueError(" | ".join(errors))
    cleaned.setdefault("IELTS_BAND", cleaned["IELTS"])
    return cleaned


def build_feature_row(stu, uni):
    gre_req = int(uni["gre_required"])
    row = {
        "IELTS":stu["IELTS"],  "IELTS_BAND":stu.get("IELTS_BAND", stu["IELTS"]),
        "TOEFL":stu["TOEFL"],  "DUOLINGO":stu["DUOLINGO"],
        "GRE_SCORE":stu["GRE_SCORE"],  "HAS_GRE":int(stu["GRE_SCORE"]>0),
        "WORK_EXP":stu["WORK_EXP"],    "CGPA":stu["CGPA"],
        "CGPA_GAP":     stu["CGPA"]     - uni["min_cgpa"],
        "IELTS_GAP":    stu["IELTS"]    - uni["min_ielts"],
        "TOEFL_GAP":    stu["TOEFL"]    - uni["toefl"],
        "DUOLINGO_GAP": stu["DUOLINGO"] - uni["duolingo"],
        "GRE_GAP":      (stu["GRE_SCORE"] - uni["gre_score"]) if gre_req else 0,
        "WORK_EXP_GAP": stu["WORK_EXP"] - uni["work_exp"],
        "UNI_IELTS":uni["min_ielts"],    "UNI_IELTS_BAND":uni["ielts_band"],
        "UNI_TOEFL":uni["toefl"],        "UNI_DUOLINGO":uni["duolingo"],
        "UNI_GRE":uni["gre_score"],      "UNI_GRE_REQ":gre_req,
        "UNI_WORKEXP":uni["work_exp"],   "UNI_CGPA":uni["min_cgpa"],
        "UNI_RANK":uni["qs_rank"],       "UNI_RANK_SCORE":uni["rank_score"],
        "UNI_RANK_TIER":uni["rank_tier"],"UNI_STRICTNESS":uni["strictness_score"],
        "UNI_TUITION":uni["tuition_usd"],
    }
    return np.array([row[f] for f in FEATURE_COLS], dtype=np.float32)


def get_missing_requirements(stu, uni):
    gaps = []
    if stu["CGPA"]     < uni["min_cgpa"]:  gaps.append(f"CGPA {stu['CGPA']:.2f} < required {uni['min_cgpa']:.1f}")
    if stu["IELTS"]    < uni["min_ielts"]: gaps.append(f"IELTS {stu['IELTS']} < required {uni['min_ielts']}")
    if stu["TOEFL"]    < uni["toefl"]:     gaps.append(f"TOEFL {stu['TOEFL']} < required {uni['toefl']}")
    if stu["DUOLINGO"] < uni["duolingo"]:  gaps.append(f"Duolingo {stu['DUOLINGO']} < required {uni['duolingo']}")
    if uni["gre_required"] and stu["GRE_SCORE"] < uni["gre_score"]:
        gaps.append(f"GRE {stu['GRE_SCORE']} < required {uni['gre_score']}")
    if uni["work_exp"] > 0 and stu["WORK_EXP"] < uni["work_exp"]:
        gaps.append(f"Work exp {stu['WORK_EXP']}yr < required {uni['work_exp']}yr")
    return gaps


def effective_gap_count(gaps):
    """
    Language tests (IELTS/TOEFL/Duolingo) are alternatives — student only needs ONE.
    So missing all three counts as ONE language gap, not three separate gaps.
    """
    lang_gaps  = [g for g in gaps if any(t in g for t in ["TOEFL","IELTS","Duolingo"])]
    other_gaps = [g for g in gaps if g not in lang_gaps]
    return len(other_gaps) + (1 if lang_gaps else 0)


def rule_based_prob(stu, uni, gaps):
    """
    Sigmoid-gap probability. Uses best language score (since tests are alternatives).
    Penalties applied per effective gap, not per raw gap.
    """
    def sig(gap, scale):
        return 1.0 / (1.0 + np.exp(-gap / scale))

    # Best available language score — whichever test the student performs best on
    s_lang = max(
        sig(stu["IELTS"]    - uni["min_ielts"], 0.35),
        sig(stu["TOEFL"]    - uni["toefl"],     3.50),
        sig(stu["DUOLINGO"] - uni["duolingo"],  3.50),
    )
    s_cgpa = sig(stu["CGPA"] - uni["min_cgpa"], 0.20)

    gre_req = uni["gre_required"] and uni["gre_score"] > 0
    if gre_req:
        s_gre = 0.05 if stu["GRE_SCORE"] == 0 else sig(stu["GRE_SCORE"] - uni["gre_score"], 7.0)
    else:
        s_gre = 0.60 + (0.10 if stu["GRE_SCORE"] >= 310 else 0.03 if stu["GRE_SCORE"] > 0 else 0.0)

    if uni["work_exp"] > 0:
        s_we = sig(stu["WORK_EXP"] - uni["work_exp"], 0.7)
    else:
        s_we = 0.60 + (0.08 if stu["WORK_EXP"] >= 2 else 0.03 if stu["WORK_EXP"] >= 1 else 0.0)

    score = s_cgpa * 0.35 + s_lang * 0.35 + s_gre * 0.18 + s_we * 0.12

    # Rank difficulty
    rank = int(uni["qs_rank"])
    if   rank <= 50:  score -= 0.13
    elif rank <= 100: score -= 0.08
    elif rank <= 200: score -= 0.04
    elif rank <= 500: score -= 0.01

    # Penalties use effective gaps (language = 1 gap max)
    lang_gaps  = [g for g in gaps if any(t in g for t in ["TOEFL","IELTS","Duolingo"])]
    other_gaps = [g for g in gaps if g not in lang_gaps]
    eff_gaps   = other_gaps + (lang_gaps[:1] if lang_gaps else [])

    for g in eff_gaps:
        if   "CGPA"     in g: score -= 0.15
        elif "IELTS"    in g: score -= 0.11
        elif "TOEFL"    in g: score -= 0.11
        elif "Duolingo" in g: score -= 0.11
        elif "GRE"      in g: score -= 0.11
        elif "Work"     in g: score -= 0.06

    n_eff    = len(eff_gaps)
    ceiling  = 0.87 if n_eff == 0 else (0.58 if n_eff == 1 else (0.36 if n_eff == 2 else 0.20))
    return float(np.clip(score, 0.05, ceiling))


def classify_tier(prob, gaps):
    n_eff = effective_gap_count(gaps)
    if   prob >= 0.65 and n_eff == 0: return "✅ Safe"
    elif prob >= 0.42:                 return "🎯 Match"
    elif prob >= 0.22 and n_eff <= 1:  return "🌟 Dream"
    else:                              return "❌ Unlikely"


def predict(user, top_k=10, country_filter=None, max_tuition=None, tier_filter=None):
    user = validate_and_clamp(user)
    results = []

    for _, uni in df_uni.iterrows():
        if country_filter and str(country_filter).strip():
            if uni["country"].lower() != str(country_filter).strip().lower():
                continue
        if max_tuition and int(max_tuition) > 0:
            if uni["tuition_usd"] > max_tuition:
                continue

        gaps = get_missing_requirements(user, uni)
        prob = rule_based_prob(user, uni, gaps)

        if prob < 0.08:
            continue

        tier     = classify_tier(prob, gaps)
        n_eff    = effective_gap_count(gaps)

        # ── Within-tier ranking score ──────────────────────────────────
        # Inside each tier, better-ranked unis with higher probability sort first.
        # rank_score is 0→1 where 1 = best QS rank.
        cgpa_gap     = user["CGPA"] - float(uni["min_cgpa"])
        cgpa_comfort = float(np.clip(cgpa_gap / 0.4, -1, 1))

        within_tier_score = float(
            0.50 * prob +
            0.35 * float(uni["rank_score"]) +
            0.15 * max(cgpa_comfort, 0)
        )

        emoji = COUNTRY_EMOJI.get(str(uni["country"]), "🌍")
        results.append({
            "university":           str(uni["university_name"]),
            "country":              f"{emoji} {uni['country']}",
            "qs_rank":              int(uni["qs_rank"]),
            "eligibility_prob":     round(prob, 4),
            "within_tier_score":    round(within_tier_score, 4),
            "tier":                 tier,
            "tier_order":           TIER_ORDER[tier],    # for sorting, not shown to user
            "tuition_usd":          int(uni["tuition_usd"]),
            "gre_required":         bool(int(uni["gre_required"])),
            "missing_requirements": gaps,
        })

    if not results:
        return [{
            "university":"No universities match your profile/filters.",
            "country":"—","qs_rank":0,"eligibility_prob":0.0,
            "within_tier_score":0.0,"tier":"❌ Unlikely","tier_order":3,
            "tuition_usd":0,"gre_required":False,
            "missing_requirements":["Profile does not meet minimum entry requirements."],
        }]

    # ── Sort: tier first (Safe → Match → Dream → Unlikely),
    #         then within each tier by within_tier_score descending,
    #         then by QS rank ascending as final tiebreaker
    results.sort(key=lambda x: (x["tier_order"], -x["within_tier_score"], x["qs_rank"]))

    # Remove internal sorting key before returning
    for r in results:
        r.pop("tier_order", None)

    if tier_filter and str(tier_filter).strip():
        results = [r for r in results if str(tier_filter).strip().lower() in r["tier"].lower()]

    return results[:top_k]


print("✅ Prediction engine v2.6 loaded!")
print("   Sorting: Safe → Match → Dream → Unlikely")
print("   Within each tier: best prob + rank first")
print("   Language tests treated as alternatives (IELTS OR TOEFL OR Duolingo)")

✅ Prediction engine v2.6 loaded!
   Sorting: Safe → Match → Dream → Unlikely
   Within each tier: best prob + rank first
   Language tests treated as alternatives (IELTS OR TOEFL OR Duolingo)


In [23]:
print("=" * 65)
print("TEST 1 — STRONG STUDENT (CGPA 3.7, IELTS 7.5, GRE 320)")
print("=" * 65)
strong = predict({"IELTS":7.5,"TOEFL":105,"DUOLINGO":130,
                  "GRE_SCORE":320,"WORK_EXP":2,"CGPA":3.7}, top_k=10)
df_s = pd.DataFrame(strong)[["qs_rank","university","country","tier","eligibility_prob","tuition_usd","missing_requirements"]]
print(f"Prob range: {df_s['eligibility_prob'].min():.3f} – {df_s['eligibility_prob'].max():.3f}")
df_s

TEST 1 — STRONG STUDENT (CGPA 3.7, IELTS 7.5, GRE 320)
Prob range: 0.836 – 0.870


,qs_rank,university,country,tier,eligibility_prob,tuition_usd,missing_requirements
0,82,Adelaide University,🇦🇺 Australia,✅ Safe,0.8451,27000,[]
1,112,Qatar University,🇶🇦 Qatar,✅ Safe,0.8573,10000,[]
2,96,University of Technology Sydney,🇦🇺 Australia,✅ Safe,0.8450,28000,[]
3,94,University of Alberta,🇨🇦 Canada,✅ Safe,0.8423,18000,[]
4,125,RMIT University,🇦🇺 Australia,✅ Safe,0.8571,26000,[]
5,151,Western University,🇨🇦 Canada,✅ Safe,0.8700,18000,[]
6,130,Humboldt-Universität zu Berlin,🇩🇪 Germany,✅ Safe,0.8571,700,[]
7,105,Rheinisch-Westfälische Technische Hochschule A...,🇩🇪 Germany,✅ Safe,0.8399,700,[]
8,110,Queen Mary University of London (QMUL),🇬🇧 UK,✅ Safe,0.8363,26000,[]
9,168,University of Montreal,🇨🇦 Canada,✅ Safe,0.8700,14000,[]


In [24]:
print("=" * 65)
print("TEST 2 — AVERAGE STUDENT (CGPA 3.0, IELTS 6.5, no GRE)")
print("=" * 65)
avg = predict({"IELTS":6.5,"TOEFL":88,"DUOLINGO":115,
               "GRE_SCORE":0,"WORK_EXP":1,"CGPA":3.0}, top_k=10)
df_a = pd.DataFrame(avg)[["qs_rank","university","country","tier","eligibility_prob","tuition_usd","missing_requirements"]]
print(f"Prob range: {df_a['eligibility_prob'].min():.3f} – {df_a['eligibility_prob'].max():.3f}")
df_a

TEST 2 — AVERAGE STUDENT (CGPA 3.0, IELTS 6.5, no GRE)
Prob range: 0.723 – 0.760


,qs_rank,university,country,tier,eligibility_prob,tuition_usd,missing_requirements
0,226,Queensland University of Technology,🇦🇺 Australia,✅ Safe,0.7605,24000,[]
1,227,The University of Newcastle,🇦🇺 Australia,✅ Safe,0.7605,22000,[]
2,310,"City St George’s, University of London",🇬🇧 UK,✅ Safe,0.7605,20000,[]
3,314,University of Tasmania,🇦🇺 Australia,✅ Safe,0.7605,18000,[]
4,374,Oxford Brookes University,🇬🇧 UK,✅ Safe,0.7605,16000,[]
5,395,Aston University,🇬🇧 UK,✅ Safe,0.7605,18000,[]
6,395,Ruhr-Universität Bochum,🇩🇪 Germany,✅ Safe,0.7605,700,[]
7,400,Western Sydney University,🇦🇺 Australia,✅ Safe,0.7605,19000,[]
8,215,Eberhard Karls Universität Tübingen,🇩🇪 Germany,✅ Safe,0.7225,700,[]
9,410,University of Southern Queensland,🇦🇺 Australia,✅ Safe,0.7605,17000,[]


In [25]:
print("=" * 60)
print("TEST 3: WEAK STUDENT — Germany only, budget < $5000")
print("=" * 60)

weak = predict({
    "IELTS": 6.0, "TOEFL": 78, "DUOLINGO": 100,
    "GRE_SCORE": 0, "WORK_EXP": 0, "CGPA": 2.8
}, top_k=10, country_filter="Germany", max_tuition=5000)

pd.DataFrame(weak)[[
    "qs_rank", "university", "country", "tier",
    "eligibility_prob", "tuition_usd", "missing_requirements"
]]

TEST 3: WEAK STUDENT — Germany only, budget < $5000


,qs_rank,university,country,tier,eligibility_prob,tuition_usd,missing_requirements
0,395,Ruhr-Universität Bochum,🇩🇪 Germany,🌟 Dream,0.4100,700,"[TOEFL 78 < required 79, Duolingo 100 < requir..."
1,215,Eberhard Karls Universität Tübingen,🇩🇪 Germany,❌ Unlikely,0.1923,700,"[CGPA 2.80 < required 2.9, IELTS 6.0 < require..."
2,243,Georg-August-Universität Göttingen,🇩🇪 Germany,❌ Unlikely,0.1791,700,"[CGPA 2.80 < required 3.0, TOEFL 78 < required..."
3,193,Universität Hamburg,🇩🇪 Germany,❌ Unlikely,0.1491,700,"[CGPA 2.80 < required 3.0, TOEFL 78 < required..."
4,535,Universität Leipzig,🇩🇪 Germany,❌ Unlikely,0.2271,700,"[CGPA 2.80 < required 2.9, TOEFL 78 < required..."
5,487,Technische Universität Bergakademie Freiberg,🇩🇪 Germany,❌ Unlikely,0.1791,700,"[CGPA 2.80 < required 3.0, TOEFL 78 < required..."
6,530,Universität Bremen,🇩🇪 Germany,❌ Unlikely,0.1891,700,"[CGPA 2.80 < required 3.0, TOEFL 78 < required..."
7,546,Universität Ulm,🇩🇪 Germany,❌ Unlikely,0.1891,700,"[CGPA 2.80 < required 3.0, TOEFL 78 < required..."
8,416,Julius-Maximilians-Universität Würzburg,🇩🇪 Germany,❌ Unlikely,0.1098,700,"[CGPA 2.80 < required 2.9, IELTS 6.0 < require..."
9,496,Justus-Liebig-Universität Gießen,🇩🇪 Germany,❌ Unlikely,0.1543,700,"[CGPA 2.80 < required 3.0, IELTS 6.0 < require..."


In [26]:
print("=" * 65)
print("TEST 3 — WEAK STUDENT (CGPA 2.8, IELTS 6.0)")
print("Expected: lower-ranked unis with modest probabilities")
print("=" * 65)
weak = predict({"IELTS":6.0,"TOEFL":72,"DUOLINGO":95,
                "GRE_SCORE":0,"WORK_EXP":0,"CGPA":2.8}, top_k=15)
df_w = pd.DataFrame(weak)[["qs_rank","university","country","tier","eligibility_prob","tuition_usd","missing_requirements"]]
print(f"Total matches: {len(weak)}")
for tier in ["✅ Safe","🎯 Match","🌟 Dream","❌ Unlikely"]:
    print(f"  {tier}: {sum(1 for r in weak if r['tier']==tier)}")
print(f"QS rank range: {df_w['qs_rank'].min()} – {df_w['qs_rank'].max()}")
print(f"Prob range: {df_w['eligibility_prob'].min():.3f} – {df_w['eligibility_prob'].max():.3f}")
df_w

TEST 3 — WEAK STUDENT (CGPA 2.8, IELTS 6.0)
Expected: lower-ranked unis with modest probabilities
Total matches: 15
  ✅ Safe: 0
  🎯 Match: 3
  🌟 Dream: 12
  ❌ Unlikely: 0
QS rank range: 112 – 1201
Prob range: 0.303 – 0.637


,qs_rank,university,country,tier,eligibility_prob,tuition_usd,missing_requirements
0,1001,University of Wolverhampton,🇬🇧 UK,🎯 Match,0.6373,13000,[]
1,1201,The University of Northampton,🇬🇧 UK,🎯 Match,0.5300,13000,[]
2,1201,Canterbury Christ Church University,🇬🇧 UK,🎯 Match,0.5300,14000,[]
3,112,Qatar University,🇶🇦 Qatar,🌟 Dream,0.3665,10000,[CGPA 2.80 < required 3.0]
4,227,The University of Newcastle,🇦🇺 Australia,🌟 Dream,0.4100,22000,"[TOEFL 72 < required 79, Duolingo 95 < require..."
5,310,"City St George’s, University of London",🇬🇧 UK,🌟 Dream,0.4100,20000,"[TOEFL 72 < required 79, Duolingo 95 < require..."
6,314,University of Tasmania,🇦🇺 Australia,🌟 Dream,0.4100,18000,"[TOEFL 72 < required 79, Duolingo 95 < require..."
7,374,Oxford Brookes University,🇬🇧 UK,🌟 Dream,0.4100,16000,"[TOEFL 72 < required 79, Duolingo 95 < require..."
8,395,Aston University,🇬🇧 UK,🌟 Dream,0.4100,18000,"[TOEFL 72 < required 80, Duolingo 95 < require..."
9,395,Ruhr-Universität Bochum,🇩🇪 Germany,🌟 Dream,0.4100,700,"[TOEFL 72 < required 79, Duolingo 95 < require..."


In [27]:
print("=" * 65)
print("TEST 4 — INPUT VALIDATION")
print("=" * 65)
tests = [
    ({"IELTS":7.0,"TOEFL":135,"DUOLINGO":115,"GRE_SCORE":0,"WORK_EXP":1,"CGPA":3.5}, "TOEFL > 120"),
    ({"IELTS":7.0,"TOEFL":100,"DUOLINGO":115,"GRE_SCORE":0,"WORK_EXP":1,"CGPA":4.8}, "CGPA > 4.0"),
    ({"IELTS":11.0,"TOEFL":100,"DUOLINGO":115,"GRE_SCORE":0,"WORK_EXP":1,"CGPA":3.2},"IELTS > 9.0"),
    ({"IELTS":6.5,"TOEFL":88,"DUOLINGO":115,"GRE_SCORE":0,"WORK_EXP":1,"CGPA":3.0},  "Valid input"),
]
for stu, label in tests:
    try:
        res = predict(stu, top_k=3)
        print(f"✅ '{label}' → accepted, {len(res)} results, prob range: {min(r['eligibility_prob'] for r in res):.3f}–{max(r['eligibility_prob'] for r in res):.3f}")
    except ValueError as e:
        print(f"🚫 '{label}' → rejected: {e}")

TEST 4 — INPUT VALIDATION
🚫 'TOEFL > 120' → rejected: TOEFL must be between 0 and 120 (got 135)
🚫 'CGPA > 4.0' → rejected: CGPA must be between 0.0 and 4.0 (got 4.8)
🚫 'IELTS > 9.0' → rejected: IELTS must be between 0.0 and 9.0 (got 11.0)
✅ 'Valid input' → accepted, 3 results, prob range: 0.760–0.760


In [28]:
print("=" * 60)
print("TEST 4: AVERAGE STUDENT — Dream universities only")
print("=" * 60)

dreams = predict({
    "IELTS": 6.5, "TOEFL": 90, "DUOLINGO": 115,
    "GRE_SCORE": 305, "WORK_EXP": 1, "CGPA": 3.2
}, top_k=15, tier_filter="Dream")

pd.DataFrame(dreams)[[
    "qs_rank", "university", "country", "tier",
    "eligibility_prob", "tuition_usd", "missing_requirements"
]]

TEST 4: AVERAGE STUDENT — Dream universities only


,qs_rank,university,country,tier,eligibility_prob,tuition_usd,missing_requirements
0,36,Monash University,🇦🇺 Australia,🌟 Dream,0.3660,31000,[CGPA 3.20 < required 3.3]
1,42,The University of Queensland,🇦🇺 Australia,🌟 Dream,0.3191,30000,[CGPA 3.20 < required 3.3]
2,25,The University of Sydney,🇦🇺 Australia,🌟 Dream,0.3001,34000,[CGPA 3.20 < required 3.3]
3,191,"Queen's University, Ontario",🇨🇦 Canada,🌟 Dream,0.3953,18000,[GRE 305 < required 308]
4,22,Technical University of Munich,🇩🇪 Germany,🌟 Dream,0.2648,700,[CGPA 3.20 < required 3.3]
5,119,University of Waterloo,🇨🇦 Canada,🌟 Dream,0.3104,19000,[GRE 305 < required 315]
6,51,University of Bristol,🇬🇧 UK,🌟 Dream,0.2661,27000,[CGPA 3.20 < required 3.3]


In [29]:
print("DIAGNOSTIC — checking why Wolverhampton disappears")
print("=" * 60)

wolv = df_uni[df_uni["university_name"].str.contains("Wolverhampton", case=False)]
print("Wolverhampton requirements:")
print(wolv[["university_name","qs_rank","min_cgpa","min_ielts","toefl","duolingo","gre_required","work_exp"]].to_string())

weak_stu = {"IELTS":6.0,"TOEFL":72,"DUOLINGO":95,"GRE_SCORE":0,"WORK_EXP":0,"CGPA":2.8}
print(f"\nStudent: {weak_stu}")

if len(wolv) > 0:
    uni_row = wolv.iloc[0]
    gaps = get_missing_requirements(weak_stu, uni_row)
    prob = rule_based_prob(weak_stu, uni_row, gaps)
    print(f"\nGaps:  {gaps}")
    print(f"Prob:  {prob:.4f}")
    print(f"Tier:  {classify_tier(prob, gaps)}")
    print(f"Kept?  {prob >= 0.08}")

print("\n--- All unis weak student qualifies for (prob >= 0.08, no filter) ---")
all_results = predict(weak_stu, top_k=50)
print(f"Total: {len(all_results)}")
for r in all_results:
    print(f"  {r['qs_rank']:>5}  {r['tier']}  {r['eligibility_prob']:.3f}  {r['university'][:40]}  gaps={len(r['missing_requirements'])}")

DIAGNOSTIC — checking why Wolverhampton disappears
Wolverhampton requirements:
                 university_name  qs_rank  min_cgpa  min_ielts  toefl  duolingo  gre_required  work_exp
204  University of Wolverhampton     1001       2.8        6.0     72        90             0         0

Student: {'IELTS': 6.0, 'TOEFL': 72, 'DUOLINGO': 95, 'GRE_SCORE': 0, 'WORK_EXP': 0, 'CGPA': 2.8}

Gaps:  []
Prob:  0.6373
Tier:  🎯 Match
Kept?  True

--- All unis weak student qualifies for (prob >= 0.08, no filter) ---
Total: 50
   1001  🎯 Match  0.637  University of Wolverhampton  gaps=0
   1201  🎯 Match  0.530  The University of Northampton  gaps=0
   1201  🎯 Match  0.530  Canterbury Christ Church University  gaps=0
    112  🌟 Dream  0.366  Qatar University  gaps=1
    227  🌟 Dream  0.410  The University of Newcastle  gaps=2
    310  🌟 Dream  0.410  City St George’s, University of London  gaps=2
    314  🌟 Dream  0.410  University of Tasmania  gaps=2
    374  🌟 Dream  0.410  Oxford Brookes University

In [30]:
import subprocess, threading, time, re, os

# Kill any existing cloudflared
os.system("pkill cloudflared 2>/dev/null")
time.sleep(1)

# Download if needed
if not os.path.exists("./cloudflared"):
    os.system("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared")
    print("Downloaded cloudflared ✓")

# Remove old log
if os.path.exists("/tmp/cf.log"):
    os.remove("/tmp/cf.log")

# Launch in background, writing to log file
proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000",
     "--no-autoupdate", "--logfile", "/tmp/cf.log"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("Cloudflared started, waiting for tunnel URL", end="")
url = None
for _ in range(30):        # wait up to 15 seconds
    time.sleep(0.5)
    print(".", end="", flush=True)
    if os.path.exists("/tmp/cf.log"):
        with open("/tmp/cf.log") as f:
            log = f.read()
        found = re.findall(r"https://[a-z0-9\-]+\.trycloudflare\.com", log)
        if found:
            url = found[0]
            break

print()
if url:
    print(f"\n🌐 Live URL : {url}")
    print(f"📖 Swagger  : {url}/docs")
    print(f"📊 Stats    : {url}/stats")
else:
    print("❌ URL not found. Check /tmp/cf.log:")
    os.system("tail -20 /tmp/cf.log")

Cloudflared started, waiting for tunnel URL........

🌐 Live URL : https://moved-striking-limits-recent.trycloudflare.com
📖 Swagger  : https://moved-striking-limits-recent.trycloudflare.com/docs
📊 Stats    : https://moved-striking-limits-recent.trycloudflare.com/stats


In [33]:
print("=" * 65)
print("TEST 1 — STRONG STUDENT (CGPA 3.7, IELTS 7.5, GRE 320)")
print("=" * 65)
strong = predict({"IELTS":7.5,"TOEFL":105,"DUOLINGO":130,
                  "GRE_SCORE":320,"WORK_EXP":2,"CGPA":3.7}, top_k=10)
df_s = pd.DataFrame(strong)[["qs_rank","university","country","tier",
                               "eligibility_prob","tuition_usd","missing_requirements"]]
print(f"Prob range: {df_s['eligibility_prob'].min():.3f} – {df_s['eligibility_prob'].max():.3f}")
df_s

TEST 1 — STRONG STUDENT (CGPA 3.7, IELTS 7.5, GRE 320)
Prob range: 0.836 – 0.870


,qs_rank,university,country,tier,eligibility_prob,tuition_usd,missing_requirements
0,82,Adelaide University,🇦🇺 Australia,✅ Safe,0.8451,27000,[]
1,112,Qatar University,🇶🇦 Qatar,✅ Safe,0.8573,10000,[]
2,96,University of Technology Sydney,🇦🇺 Australia,✅ Safe,0.8450,28000,[]
3,94,University of Alberta,🇨🇦 Canada,✅ Safe,0.8423,18000,[]
4,125,RMIT University,🇦🇺 Australia,✅ Safe,0.8571,26000,[]
5,151,Western University,🇨🇦 Canada,✅ Safe,0.8700,18000,[]
6,130,Humboldt-Universität zu Berlin,🇩🇪 Germany,✅ Safe,0.8571,700,[]
7,105,Rheinisch-Westfälische Technische Hochschule A...,🇩🇪 Germany,✅ Safe,0.8399,700,[]
8,110,Queen Mary University of London (QMUL),🇬🇧 UK,✅ Safe,0.8363,26000,[]
9,168,University of Montreal,🇨🇦 Canada,✅ Safe,0.8700,14000,[]


In [34]:
print("=" * 65)
print("TEST 2 — AVERAGE STUDENT (CGPA 3.0, IELTS 6.5, no GRE)")
print("=" * 65)
avg = predict({"IELTS":6.5,"TOEFL":88,"DUOLINGO":115,
               "GRE_SCORE":0,"WORK_EXP":1,"CGPA":3.0}, top_k=10)
df_a = pd.DataFrame(avg)[["qs_rank","university","country","tier",
                            "eligibility_prob","tuition_usd","missing_requirements"]]
print(f"Prob range: {df_a['eligibility_prob'].min():.3f} – {df_a['eligibility_prob'].max():.3f}")
df_a

TEST 2 — AVERAGE STUDENT (CGPA 3.0, IELTS 6.5, no GRE)
Prob range: 0.723 – 0.760


,qs_rank,university,country,tier,eligibility_prob,tuition_usd,missing_requirements
0,226,Queensland University of Technology,🇦🇺 Australia,✅ Safe,0.7605,24000,[]
1,227,The University of Newcastle,🇦🇺 Australia,✅ Safe,0.7605,22000,[]
2,310,"City St George’s, University of London",🇬🇧 UK,✅ Safe,0.7605,20000,[]
3,314,University of Tasmania,🇦🇺 Australia,✅ Safe,0.7605,18000,[]
4,374,Oxford Brookes University,🇬🇧 UK,✅ Safe,0.7605,16000,[]
5,395,Aston University,🇬🇧 UK,✅ Safe,0.7605,18000,[]
6,395,Ruhr-Universität Bochum,🇩🇪 Germany,✅ Safe,0.7605,700,[]
7,400,Western Sydney University,🇦🇺 Australia,✅ Safe,0.7605,19000,[]
8,215,Eberhard Karls Universität Tübingen,🇩🇪 Germany,✅ Safe,0.7225,700,[]
9,410,University of Southern Queensland,🇦🇺 Australia,✅ Safe,0.7605,17000,[]


In [35]:
print("=" * 65)
print("TEST 3 — WEAK STUDENT (CGPA 2.8, IELTS 6.0, no GRE)")
print("Expected: Match unis first, then Dreams, no elite unis")
print("=" * 65)
weak = predict({"IELTS":6.0,"TOEFL":72,"DUOLINGO":95,
                "GRE_SCORE":0,"WORK_EXP":0,"CGPA":2.8}, top_k=12)
df_w = pd.DataFrame(weak)[["qs_rank","university","country","tier",
                             "eligibility_prob","tuition_usd","missing_requirements"]]
print(f"Total matches: {len(weak)}")
for tier in ["✅ Safe","🎯 Match","🌟 Dream","❌ Unlikely"]:
    print(f"  {tier}: {sum(1 for r in weak if r['tier']==tier)}")
print(f"Prob range: {df_w['eligibility_prob'].min():.3f} – {df_w['eligibility_prob'].max():.3f}")
df_w

TEST 3 — WEAK STUDENT (CGPA 2.8, IELTS 6.0, no GRE)
Expected: Match unis first, then Dreams, no elite unis
Total matches: 12
  ✅ Safe: 0
  🎯 Match: 3
  🌟 Dream: 9
  ❌ Unlikely: 0
Prob range: 0.303 – 0.637


,qs_rank,university,country,tier,eligibility_prob,tuition_usd,missing_requirements
0,1001,University of Wolverhampton,🇬🇧 UK,🎯 Match,0.6373,13000,[]
1,1201,The University of Northampton,🇬🇧 UK,🎯 Match,0.5300,13000,[]
2,1201,Canterbury Christ Church University,🇬🇧 UK,🎯 Match,0.5300,14000,[]
3,112,Qatar University,🇶🇦 Qatar,🌟 Dream,0.3665,10000,[CGPA 2.80 < required 3.0]
4,227,The University of Newcastle,🇦🇺 Australia,🌟 Dream,0.4100,22000,"[TOEFL 72 < required 79, Duolingo 95 < require..."
5,310,"City St George’s, University of London",🇬🇧 UK,🌟 Dream,0.4100,20000,"[TOEFL 72 < required 79, Duolingo 95 < require..."
6,314,University of Tasmania,🇦🇺 Australia,🌟 Dream,0.4100,18000,"[TOEFL 72 < required 79, Duolingo 95 < require..."
7,374,Oxford Brookes University,🇬🇧 UK,🌟 Dream,0.4100,16000,"[TOEFL 72 < required 79, Duolingo 95 < require..."
8,395,Aston University,🇬🇧 UK,🌟 Dream,0.4100,18000,"[TOEFL 72 < required 80, Duolingo 95 < require..."
9,395,Ruhr-Universität Bochum,🇩🇪 Germany,🌟 Dream,0.4100,700,"[TOEFL 72 < required 79, Duolingo 95 < require..."


In [36]:
print("=" * 65)
print("TEST 4 — FILTERS (Germany only, budget $5000, average student)")
print("=" * 65)
filtered = predict({"IELTS":6.5,"TOEFL":88,"DUOLINGO":115,
                    "GRE_SCORE":0,"WORK_EXP":1,"CGPA":3.0},
                   top_k=10, country_filter="Germany", max_tuition=5000)
pd.DataFrame(filtered)[["qs_rank","university","country","tier",
                          "eligibility_prob","tuition_usd","missing_requirements"]]

TEST 4 — FILTERS (Germany only, budget $5000, average student)


,qs_rank,university,country,tier,eligibility_prob,tuition_usd,missing_requirements
0,395,Ruhr-Universität Bochum,🇩🇪 Germany,✅ Safe,0.7605,700,[]
1,215,Eberhard Karls Universität Tübingen,🇩🇪 Germany,✅ Safe,0.7225,700,[]
2,201,Universität Freiburg,🇩🇪 Germany,✅ Safe,0.6796,700,[]
3,207,Rheinische Friedrich-Wilhelms-Universität Bonn,🇩🇪 Germany,✅ Safe,0.6796,700,[]
4,416,Julius-Maximilians-Universität Würzburg,🇩🇪 Germany,✅ Safe,0.7225,700,[]
5,218,Technische Universität Dresden,🇩🇪 Germany,✅ Safe,0.6796,700,[]
6,232,Universität Erlangen-Nürnberg,🇩🇪 Germany,✅ Safe,0.6796,700,[]
7,440,Universität Konstanz,🇩🇪 Germany,✅ Safe,0.7225,700,[]
8,243,Georg-August-Universität Göttingen,🇩🇪 Germany,✅ Safe,0.6796,700,[]
9,253,Technische Universität Darmstadt,🇩🇪 Germany,✅ Safe,0.6796,700,[]


In [37]:
print("=" * 65)
print("TEST 5 — INPUT VALIDATION")
print("=" * 65)
tests = [
    ({"IELTS":7.0,"TOEFL":135,"DUOLINGO":115,"GRE_SCORE":0,"WORK_EXP":1,"CGPA":3.5}, "TOEFL > 120"),
    ({"IELTS":7.0,"TOEFL":100,"DUOLINGO":115,"GRE_SCORE":0,"WORK_EXP":1,"CGPA":4.8}, "CGPA > 4.0"),
    ({"IELTS":11.0,"TOEFL":100,"DUOLINGO":115,"GRE_SCORE":0,"WORK_EXP":1,"CGPA":3.2},"IELTS > 9.0"),
    ({"IELTS":6.5,"TOEFL":88,"DUOLINGO":115,"GRE_SCORE":0,"WORK_EXP":1,"CGPA":3.0},  "Valid input"),
]
for stu, label in tests:
    try:
        res = predict(stu, top_k=3)
        probs = [r['eligibility_prob'] for r in res]
        print(f"✅ '{label}' → accepted, {len(res)} results, prob range: {min(probs):.3f}–{max(probs):.3f}")
    except ValueError as e:
        print(f"🚫 '{label}' → rejected: {e}")

TEST 5 — INPUT VALIDATION
🚫 'TOEFL > 120' → rejected: TOEFL must be between 0 and 120 (got 135)
🚫 'CGPA > 4.0' → rejected: CGPA must be between 0.0 and 4.0 (got 4.8)
🚫 'IELTS > 9.0' → rejected: IELTS must be between 0.0 and 9.0 (got 11.0)
✅ 'Valid input' → accepted, 3 results, prob range: 0.760–0.760


In [38]:
from fastapi import FastAPI, Query, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field, field_validator
from typing import Optional
import nest_asyncio, uvicorn, json, numpy as np

class NumpySafeEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):  return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.bool_):    return bool(obj)
        if isinstance(obj, np.ndarray):  return obj.tolist()
        return super().default(obj)

app = FastAPI(
    title="🎓 Mashwara-e-Taleem — University Eligibility AI",
    description=(
        "Intelligent university recommender for Pakistani students.\n\n"
        "Results sorted by **tier first** (Safe → Match → Dream → Unlikely), "
        "then by rank + probability within each tier.\n\n"
        "**Score limits enforced:** IELTS 0–9 | TOEFL 0–120 | "
        "Duolingo 10–160 | GRE 0–340 | CGPA 0–4.0"
    ),
    version="2.6"
)

class StudentInput(BaseModel):
    IELTS:          float         = Field(...,  ge=0.0, le=9.0,  description="IELTS overall (0–9)")
    TOEFL:          int           = Field(...,  ge=0,   le=120,  description="TOEFL iBT (0–120)")
    DUOLINGO:       int           = Field(110,  ge=10,  le=160,  description="Duolingo (10–160)")
    GRE_SCORE:      int           = Field(0,    ge=0,   le=340,  description="GRE total (0 = not taken)")
    WORK_EXP:       int           = Field(0,    ge=0,   le=40,   description="Years of work experience")
    CGPA:           float         = Field(...,  ge=0.0, le=4.0,  description="CGPA on 4.0 scale")
    top_k:          int           = Field(10,   ge=1,   le=50,   description="Max results")
    country_filter: Optional[str] = Field(None, description="UK / Germany / Australia / Canada / Qatar")
    max_tuition:    Optional[int] = Field(None, ge=0,   description="Max tuition in USD")
    tier_filter:    Optional[str] = Field(None, description="Safe / Match / Dream / Unlikely")

    @field_validator("country_filter", "tier_filter", mode="before")
    @classmethod
    def empty_to_none(cls, v):
        return None if isinstance(v, str) and v.strip() == "" else v

    @field_validator("country_filter")
    @classmethod
    def check_country(cls, v):
        if v is None: return v
        if v.strip().lower() not in {"uk","germany","australia","canada","qatar"}:
            raise ValueError("Must be one of: UK, Germany, Australia, Canada, Qatar")
        return v.strip().title()

    @field_validator("tier_filter")
    @classmethod
    def check_tier(cls, v):
        if v is None: return v
        if v.strip().lower() not in {"safe","match","dream","unlikely"}:
            raise ValueError("Must be one of: Safe, Match, Dream, Unlikely")
        return v.strip().capitalize()


@app.get("/")
def home():
    return JSONResponse(content={
        "status":              "🟢 Running",
        "service":             "Mashwara-e-Taleem University AI v2.6",
        "universities_loaded": int(len(df_uni)),
        "model":               "Rule-Calibrated Probabilistic Recommender",
        "sorting":             "Tier-first (Safe→Match→Dream→Unlikely), then rank+prob within tier",
        "score_limits":        {"IELTS":"0–9","TOEFL":"0–120","DUOLINGO":"10–160",
                                "GRE":"0–340","CGPA":"0–4.0","WORK_EXP":"0–40 yrs"},
        "tiers": {
            "✅ Safe":      "Meets all requirements, high probability",
            "🎯 Match":     "High probability, apply with confidence",
            "🌟 Dream":     "Ambitious but achievable with 1 gap",
            "❌ Unlikely":  "Missing multiple requirements"
        }
    })


@app.post("/predict")
def api_predict(student: StudentInput):
    data           = student.model_dump()
    top_k          = data.pop("top_k")
    country_filter = data.pop("country_filter")
    max_tuition    = data.pop("max_tuition")
    tier_filter    = data.pop("tier_filter")

    try:
        results = predict(data, top_k=top_k, country_filter=country_filter,
                          max_tuition=max_tuition, tier_filter=tier_filter)
    except ValueError as e:
        raise HTTPException(status_code=422, detail=str(e))

    # Rename within_tier_score → final_score for clean API output
    for r in results:
        if "within_tier_score" in r:
            r["final_score"] = r.pop("within_tier_score")

    clean = json.loads(json.dumps(results, cls=NumpySafeEncoder))

    tiers = {"✅ Safe":0, "🎯 Match":0, "🌟 Dream":0, "❌ Unlikely":0}
    for r in clean:
        t = r.get("tier", "❌ Unlikely")
        tiers[t] = tiers.get(t, 0) + 1

    return JSONResponse(content={
        "total_results": len(clean),
        "tier_summary":  tiers,
        "results":       clean,
    })


@app.get("/universities")
def list_universities(country: Optional[str] = Query(None)):
    unis = df_uni.copy()
    if country and country.strip():
        unis = unis[unis["country"].str.lower() == country.strip().lower()]
    records = json.loads(json.dumps(
        unis[["university_name","country","qs_rank","min_cgpa","min_ielts",
              "toefl","duolingo","gre_required","tuition_usd"]]
        .sort_values("qs_rank").to_dict(orient="records"),
        cls=NumpySafeEncoder
    ))
    return JSONResponse(content={"total": len(records), "universities": records})


@app.get("/stats")
def stats():
    return JSONResponse(content=json.loads(json.dumps({
        "total_universities": int(len(df_uni)),
        "countries":          df_uni["country"].value_counts().to_dict(),
        "qs_rank_range":      {"min":int(df_uni["qs_rank"].min()), "max":int(df_uni["qs_rank"].max())},
        "tuition_range":      {"min":int(df_uni["tuition_usd"].min()), "max":int(df_uni["tuition_usd"].max())},
        "avg_min_cgpa":       round(float(df_uni["min_cgpa"].mean()), 2),
        "avg_min_ielts":      round(float(df_uni["min_ielts"].mean()), 2),
        "gre_required_count": int(df_uni["gre_required"].sum()),
    }, cls=NumpySafeEncoder)))


nest_asyncio.apply()
print("🚀 FastAPI v2.6 ready — run Cell 11 first to get your tunnel URL")
config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, log_level="warning")
server = uvicorn.Server(config)
await server.serve()

🚀 FastAPI v2.6 ready — run Cell 11 first to get your tunnel URL
